# LAB: Practice with Tavily + LangChain Agents



### Objective

Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:

- Load and configure a LangChain agent with Tavily

- Build effective prompts

- Generate structured, useful outputs from real-time data

### Setup

In [7]:
%pip install langchain langchain_community langchain_openai tiktoken tavily-python python-dotenv -q

import os
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import Tool
import tiktoken
from IPython.display import Markdown, display
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

load_dotenv()

from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
encoding = tiktoken.encoding_for_model("gpt-4o")

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    result = tavily_search.run(query)

    # Ensure result is a string - Tavily returns dict with 'snippets' sometimes
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:2000])  # leave room for GPT-4 response
    return trimmed


# LangChain tool
tools = [Tool(name="TavilySafeSearch", func=safe_search, description="Web search tool")]

# --- New LangChain 1.x agent API (replaces the deprecated initialize_agent) ---
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful research assistant with access to a web search tool. Use it to find current, accurate information before answering."
)

def run_agent(prompt: str) -> str:
    """Helper to invoke the new-style agent and extract just the final text answer."""
    result = agent.invoke({"messages": [{"role": "user", "content": prompt}]})
    return result["messages"][-1].content

### Exercise 1: AI in Healthcare



Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.



- Design a prompt that asks the agent to retrieve the most recent updates.

- Ensure the agent outputs a structured response in Markdown.

In [8]:
# Your prompt here
prompt_1 = """Search the web for the latest advancements in generative AI applied to healthcare in 2025.
Focus on real, recent developments (clinical applications, diagnostics, drug discovery, patient-facing tools).

Structure your answer in Markdown with:
- A short introduction (1-2 sentences)
- A section per major advancement, using ## headers
- For each advancement: what it is, which organization/company is behind it, and why it matters
- A closing section with sources/links found during the search

Keep the tone factual and cite where each piece of information came from."""

# Run it
response_1 = run_agent(prompt_1)

In [9]:
display(Markdown(response_1))


# Advancements in Generative AI Applied to Healthcare in 2025

Generative AI is making significant strides in healthcare, transforming various aspects from diagnostics to drug discovery. In 2025, several key advancements have emerged, showcasing the potential of AI technologies to enhance patient care and streamline medical processes.

## Medical Imaging Enhancement
Generative AI algorithms are revolutionizing medical imaging by enhancing the resolution and clarity of diagnostic images. This advancement allows healthcare professionals to interpret images with greater accuracy, leading to improved diagnostic outcomes. Companies like **Murphi AI** are at the forefront of this technology, which is crucial for accurate disease detection and treatment planning. Enhanced imaging capabilities can significantly reduce misdiagnoses and improve patient outcomes.

[Source: Murphi AI](https://murphi.ai/generative-ai-in-healthcare-use-cases-benefits-and-applications)

## Drug Discovery and Development
One of the most groundbreaking advancements in 2025 is the use of generative AI in drug discovery. **Insilico Medicine** developed Rentosertib, the first drug where both the target and compound were identified using generative AI, which received official recognition from the USAN Council. This milestone demonstrates the potential of AI to accelerate the drug development process, reducing both time and costs associated with bringing new medications to market. The ability to simulate molecular interactions and predict drug efficacy is transforming pharmaceutical research.

[Source: DelveInsight](https://www.delveinsight.com/blog/generative-ai-drug-discovery-market-impact)

## Personalized Medicine
Generative AI is also enhancing precision medicine by analyzing genetic, lifestyle, and environmental factors to create tailored treatment plans. This approach allows for hyper-personalized recommendations, improving treatment effectiveness for complex diseases. Companies are increasingly integrating AI into their workflows to provide more accurate and individualized patient care, which is essential for managing chronic conditions and optimizing therapeutic outcomes.

[Source: John Snow Labs](https://www.johnsnowlabs.com/generative-ai-healthcare)

## Clinical Decision Support
Generative AI is being utilized to improve clinical decision support systems, enabling healthcare providers to make more informed decisions based on real-time data analysis. By integrating multimodal AI models that can process text, images, and patient vitals, healthcare professionals can receive comprehensive insights that enhance diagnostic accuracy and treatment strategies. This advancement is crucial for improving patient safety and operational efficiency in healthcare settings.

[Source: MDPI](https://www.mdpi.com/2673-7426/5/3/37)

## Patient Communication Tools
Generative AI is also enhancing patient-facing tools, improving communication between healthcare providers and patients. AI-driven chatbots and virtual assistants are being deployed to answer patient queries, schedule appointments, and provide health information. This technology not only improves patient engagement but also reduces the administrative burden on healthcare staff, allowing them to focus more on direct patient care.

[Source: LinkedIn](https://www.linkedin.com/pulse/how-generative-ai-revolutionizing-drug-discovery-patient-care-5tmxc)

# Conclusion
The advancements in generative AI in healthcare in 2025 highlight the transformative potential of these technologies across various domains, including diagnostics, drug discovery, and patient care. As organizations continue to integrate AI into their practices, the future of healthcare looks promising, with improved outcomes and efficiencies on the horizon.

### Sources
- [Murphi AI](https://murphi.ai/generative-ai-in-healthcare-use-cases-benefits-and-applications)
- [DelveInsight](https://www.delveinsight.com/blog/generative-ai-drug-discovery-market-impact)
- [John Snow Labs](https://www.johnsnowlabs.com/generative-ai-healthcare)
- [MDPI](https://www.mdpi.com/2673-7426/5/3/37)
- [LinkedIn](https://www.linkedin.com/pulse/how-generative-ai-revolutionizing-drug-discovery-patient-care-5tmxc)

### Exercise 2: AI Startups Landscape



Goal: Track 2025’s top emerging AI startups and their innovations.



- Create a prompt that instructs the agent to deliver a clean Markdown summary.

- Tip: Ask for company names, product highlights, and sources.

In [10]:
# Your prompt here
prompt_2 = """Search the web for 2025's top emerging AI startups and their innovations.

Structure your answer in Markdown with:
- A short introduction (1-2 sentences)
- A bullet list of startups, each with: company name (bold), one-line product highlight, and what makes it notable
- A closing section listing the sources/links used

Focus on startups gaining real traction or funding in 2025, not just well-known incumbents."""

# Run it
response_2 = run_agent(prompt_2)

In [11]:
display(Markdown(response_2))


# Top Emerging AI Startups of 2025

As we move through 2025, several innovative AI startups are gaining traction and securing significant funding. These companies are pushing the boundaries of technology across various sectors, from healthcare to legal services, showcasing their potential to transform industries.

## Notable Startups

- **Ambience Healthcare**  
  _AI operating system for healthcare workflows_  
  Notable for reducing administrative burdens on clinicians, Ambience has raised $243M in Series C funding, indicating strong demand for its solutions in healthcare.

- **Fathom.ai**  
  _AI meeting assistant for productivity and sales_  
  This startup is gaining attention for its ability to enhance productivity in meetings, making it a valuable tool for sales teams.

- **Supio**  
  _AI platform for legal research_  
  With a $60M Series B round, Supio is revolutionizing how attorneys conduct research, streamlining workflows and improving case outcomes.

- **Centific Inc**  
  _Enterprise-grade AI and analytics solutions_  
  Also securing $60M in funding, Centific focuses on helping businesses leverage data for smarter decision-making, making it a key player in digital transformation.

- **Kilo**  
  _Open-source AI coding assistant_  
  Launched in 2025, Kilo offers over 500 models for development, attracting attention with its innovative approach to coding assistance.

- **Anysphere**  
  _Automating code industry leader_  
  Known for its AI code tool Cursor, Anysphere raised $3.5 billion to enhance its AI systems and expand its research capabilities.

- **AI Squared**  
  _Integrating AI into business applications_  
  This startup is notable for its acquisition of Multiwoven, enhancing its ability to embed AI insights directly into applications.

- **Anthropic**  
  _Owner of the Claude LLM models_  
  With a valuation exceeding $61 billion, Anthropic is a major player in the AI landscape, focusing on complex problem-solving through its advanced models.

## Sources
- [Wellows - 85 Hottest AI Startups to Watch in 2026](https://wellows.com/blog/ai-startups)
- [Greater Seattle Partners - AI Startups Funding in 2025](https://greater-seattle.com/ai-startups-funding-2025)
- [Eqvista - Top US AI Funding Rounds: $100M+ in 2025](https://eqvista.com/top-us-ai-funding)
- [StartupBlink - Top AI Startups in 2025](https://www.startupblink.com/blog/top-ai-startups)
- [CRN - The 10 Hottest AI Startup Companies Of 2025 (So Far)](https://www.crn.com/news/ai/2025/the-10-hottest-ai-startup-companies-of-2025-so-far)

### Exercice 3: Compare Two Tech Products

- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.

- Ensure the agent outputs a structured response in Markdown.






In [12]:
# Your prompt here
prompt_3 = """Search the web and compare the key features, pricing, and reviews of OpenAI's ChatGPT Team
and Anthropic's Claude Pro (or the closest comparable current plan for teams/professionals).

Structure your answer in Markdown as a comparison with:
- A short introduction (1-2 sentences)
- A markdown table comparing: Price, Key features, Target audience, User feedback/reviews
- A short closing paragraph with your take on which is better suited for which use case
- A section listing the sources/links used

Be accurate and note if pricing or plan names may have changed since your search."""

# Run it
response_3 = run_agent(prompt_3)

In [13]:
display(Markdown(response_3))


## Comparison of OpenAI's ChatGPT Team and Anthropic's Claude Pro

Both OpenAI's ChatGPT Team and Anthropic's Claude Pro are designed for professional and team use, offering advanced features tailored to enhance productivity and collaboration. Below is a comparison of their key features, pricing, and user feedback.

| Feature/Aspect         | OpenAI ChatGPT Team                          | Anthropic Claude Pro                          |
|------------------------|----------------------------------------------|-----------------------------------------------|
| **Price**              | $25/user/month (annual) or $30/user/month (monthly) | $17/month (annual) or $20/month (monthly)    |
| **Key Features**       | - Collaborative workspaces<br>- Data privacy (conversations not used for training)<br>- Admin controls<br>- Increased usage limits (100 messages/3 hours)<br>- Early access to new features | - Higher usage limits (5x more than free)<br>- Access to advanced models<br>- Enhanced reasoning and coding capabilities<br>- File uploads and longer conversations<br>- Integration with Microsoft 365 |
| **Target Audience**    | Small to mid-sized teams (2-149 users)     | Professionals and power users                 |
| **User Feedback/Reviews** | Generally positive, praised for collaboration features and data privacy; some concerns about limited enterprise-grade security | Highly rated for reasoning and context handling; users appreciate the increased productivity and advanced features |

### Conclusion
OpenAI's ChatGPT Team is better suited for small to mid-sized teams that prioritize collaboration and data privacy, making it ideal for businesses handling sensitive information. On the other hand, Anthropic's Claude Pro is tailored for professionals who require advanced reasoning capabilities and higher usage limits, making it a strong choice for individual power users or teams focused on complex tasks.

### Sources
1. [ChatGPT Team Review](https://www.linkedin.com/posts/accessible-ai_chatgpt-for-teams-is-here-heres-my-review-activity-7152271601139425280-cEU2)
2. [ChatGPT Pricing Guide](https://www.cloudeagle.ai/blogs/blog-chatgpt-pricing-guide)
3. [Claude AI Review](https://www.bleap.finance/en-us/blog/claude-ai-review)
4. [Claude Subscription Plans](https://intuitionlabs.ai/articles/claude-pricing-plans-api-costs)
5. [Introducing Claude Pro](https://www.anthropic.com/news/claude-pro)

## Bonus Task: Propose and Implement Your Own Use Case



As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.
